In [ ]:
# ===========================================
# Quantum Wavelet Attention Network (QWAN)
# ===========================================
# pip install pennylane
import numpy as np
import torch
import torch.nn as nn
import pennylane as qml

# ------------------
# Reproducibility
# ------------------
torch.manual_seed(0)
np.random.seed(0)

# -----------------------------
# Parameters
# -----------------------------
N = 256
n_qubits = 8
epochs = 100
lr = 1e-3
alpha = 0.1
beta = 0.05

dev = qml.device("default.qubit", wires=n_qubits)

# =======================================
# Synthetic ECG
# =======================================
def generate_ecg(batch_size):
    t = torch.linspace(0, 1, N)

    clean = (
        0.4 * torch.exp(-(t - 0.2)**2 / 0.002) +
        1.2 * torch.exp(-(t - 0.5)**2 / 0.0005) +
        0.6 * torch.exp(-(t - 0.75)**2 / 0.004)
    )

    clean = clean.repeat(batch_size, 1)
    noisy = clean + 0.2 * torch.randn_like(clean)
    return noisy, clean

# ======================================
# Variational Quantum Circuit
# ======================================
def vqc(weights):
    idx = 0
    for _ in range(3):
        for q in range(n_qubits):
            qml.RY(weights[idx], wires=q)
            idx += 1
            qml.RZ(weights[idx], wires=q)
            idx += 1

        for q in range(n_qubits - 1):
            qml.CNOT(wires=[q, q + 1])

# ============================================
# VQC1: Decomposition
# ============================================
@qml.qnode(dev, interface="torch")
def vqc1(x, weights):
    qml.AmplitudeEmbedding(x, wires=range(n_qubits), normalize=True)
    vqc(weights)
    return qml.probs(wires=range(n_qubits))

# ============================================
# VQC2: Reconstruction
# ============================================
@qml.qnode(dev, interface="torch")
def vqc2(z, weights):
    qml.AmplitudeEmbedding(z, wires=range(n_qubits), normalize=True)
    vqc(weights)
    return qml.probs(wires=range(n_qubits))

# ===================================
# Quantum Metrics
# ===================================
def density_matrix(p):
    p = p / torch.sum(p)
    return torch.outer(p, p)

def von_neumann_entropy(rho):
    eig = torch.linalg.eigvalsh(rho)
    eig = torch.clamp(eig.real, min=1e-8)
    return -torch.sum(eig * torch.log(eig))

# ==========================
# Multiscale split
# ==========================
def multiscale_split(z):
    n = z.shape[0]
    return z[:n // 2], z[n // 2:]

# =======================
# Temporal attention
# =======================
def temporal_attention(rho):
    n = rho.shape[0]
    O = torch.zeros(n)

    O[:n//3] = 1.0
    O[n//3:2*n//3] = 2.0
    O[2*n//3:] = 1.5

    O = torch.diag(O)
    return torch.trace(O @ rho) * rho

# =================
# Inter-scale MI
# =================
def inter_scale_mi(rho1, rho2):
    return torch.sum((rho1 - rho2) ** 2)

# =================================
# Entanglement threshold
# =================================
def entanglement_threshold(rho, lam=0.4):
    d = torch.diag(rho)
    D = torch.diag(torch.exp(-d / lam))
    return D @ rho @ D

# =============
# QWAN Model
# =============
class QWAN(nn.Module):
    def __init__(self):
        super().__init__()

        self.w_vqc1 = nn.Parameter(
            0.01 * torch.randn(3 * n_qubits * 2)
        )
        self.w_vqc2 = nn.Parameter(
            0.01 * torch.randn(3 * n_qubits * 2)
        )

    def forward(self, x):
        outputs, entropies, mis = [], [], []

        for i in range(x.shape[0]):

            # --- Decomposition ---
            z = vqc1(x[i], self.w_vqc1)

            # --- Multiscale ---
            c, d = multiscale_split(z)
            z_ms = torch.cat([c, d])

            rho1 = density_matrix(z_ms)
            entropies.append(von_neumann_entropy(rho1))

            # --- Attention ---
            rho1 = temporal_attention(rho1)
            rho1 = entanglement_threshold(rho1)

            # --- Latent ---
            z_refined = torch.diag(rho1)
            z_refined = z_refined / torch.norm(z_refined)

            # --- Reconstruction ---
            z2 = vqc2(z_refined, self.w_vqc2)

            rho2 = density_matrix(z2)
            rho2 = temporal_attention(rho2)
            rho2 = entanglement_threshold(rho2)

            mis.append(inter_scale_mi(rho1, rho2))

            recon = torch.diag(rho2)
            recon = recon / torch.norm(recon)

            outputs.append(recon)

        return (
            torch.stack(outputs),
            torch.mean(torch.stack(entropies)),
            torch.mean(torch.stack(mis)),
        )

# ================
# Training
# ================
model = QWAN()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

for epoch in range(1, epochs + 1):

    noisy, clean = generate_ecg(8)

    noisy = noisy / torch.norm(noisy, dim=1, keepdim=True)
    clean = clean / torch.norm(clean, dim=1, keepdim=True)

    optimizer.zero_grad()

    pred, ent, mi = model(noisy)

    loss_rec = torch.mean((pred - clean) ** 2)
    loss = loss_rec + alpha * ent + beta * mi

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"Total Loss: {loss.item():.6f} | "
            f"Rec: {loss_rec.item():.6f} | "
            f"Ent: {ent.item():.6f} | "
            f"MI: {mi.item():.6f}"
        )